# Dữ liệu

In [1]:
import os
import cv2
import json
import kagglehub
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

dataset_path = kagglehub.dataset_download("solesensei/solesensei_bdd100k")

print(dataset_path)

IMG_TRAIN = os.path.join(
    dataset_path,
    "bdd100k/bdd100k/images/10k/train"
)

LABEL_JSON = os.path.join(
    dataset_path,
    "bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json"
)

SEG_IMG = os.path.join(
    dataset_path,
    "bdd100k_seg/bdd100k/seg/images/train"
)

SEG_MASK = os.path.join(
    dataset_path,
    "bdd100k_seg/bdd100k/seg/color_labels/train"
)

os.makedirs("models", exist_ok=True)

Using Colab cache for faster access to the 'solesensei_bdd100k' dataset.
/kaggle/input/solesensei_bdd100k


# Dữ liệu Object Detection

In [2]:
TARGET_CLASSES = {
    "car":0,
    "bus":1,
    "truck":2,
    "person":3,
    "rider":4,
    "bike":5,
    "motor":6
}

IMG_SIZE = 96
MAX_OBJECTS = 5000

with open(LABEL_JSON,"r") as f:
    annotations = json.load(f)

train_images = set(os.listdir(IMG_TRAIN))

filtered_annotations = [
    item
    for item in annotations
    if item["name"] in train_images
]

X_obj = []
Y_obj = []

for item in tqdm(filtered_annotations):

    img_path = os.path.join(
        IMG_TRAIN,
        item["name"]
    )

    img = cv2.imread(img_path)

    if img is None:
        continue

    for obj in item["labels"]:

        cat = obj.get("category","")

        if cat not in TARGET_CLASSES:
            continue

        if "box2d" not in obj:
            continue

        box = obj["box2d"]

        x1 = max(0,int(box["x1"]))
        y1 = max(0,int(box["y1"]))
        x2 = int(box["x2"])
        y2 = int(box["y2"])

        crop = img[y1:y2,x1:x2]

        if crop.size == 0:
            continue

        crop = cv2.resize(
            crop,
            (IMG_SIZE,IMG_SIZE)
        )

        crop = cv2.cvtColor(
            crop,
            cv2.COLOR_BGR2RGB
        )

        X_obj.append(crop)
        Y_obj.append(
            TARGET_CLASSES[cat]
        )

        if len(X_obj) >= MAX_OBJECTS:
            break

    if len(X_obj) >= MAX_OBJECTS:
        break

X_obj = np.array(
    X_obj,
    dtype=np.float32
)/255.0

Y_obj = tf.keras.utils.to_categorical(
    Y_obj,
    num_classes=7
)

print(X_obj.shape)
print(Y_obj.shape)

 12%|█▏        | 357/2972 [00:02<00:17, 145.53it/s]


(5000, 96, 96, 3)
(5000, 7)


# CNN OBJECT CLASSIFIER

In [3]:
X_train_obj,X_test_obj,Y_train_obj,Y_test_obj = \
train_test_split(
    X_obj,
    Y_obj,
    test_size=0.2,
    random_state=42
)

cnn = tf.keras.Sequential([

    tf.keras.layers.Input(
        shape=(96,96,3)
    ),

    tf.keras.layers.Conv2D(
        32,3,
        activation='relu'
    ),

    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(
        64,3,
        activation='relu'
    ),

    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(
        128,3,
        activation='relu'
    ),

    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        256,
        activation='relu'
    ),

    tf.keras.layers.Dropout(
        0.5
    ),

    tf.keras.layers.Dense(
        7,
        activation='softmax'
    )
])

cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_obj = cnn.fit(
    X_train_obj,
    Y_train_obj,
    validation_split=0.2,
    epochs=3,
    batch_size=8
)

Epoch 1/3
400/400 ━━━━━━━━━━━━━━━━━━━━ 77s 177ms/step - accuracy: 0.7969 - loss: 0.7655 - val_accuracy: 0.8250 - val_loss: 0.6274
Epoch 2/3
400/400 ━━━━━━━━━━━━━━━━━━━━ 80s 171ms/step - accuracy: 0.8056 - loss: 0.6789 - val_accuracy: 0.8175 - val_loss: 0.6016
Epoch 3/3
400/400 ━━━━━━━━━━━━━━━━━━━━ 99s 214ms/step - accuracy: 0.8091 - loss: 0.6596 - val_accuracy: 0.8300 - val_loss: 0.5894


# Đánh giá Object Detection

In [4]:
pred = cnn.predict(X_test_obj)

y_true = np.argmax(
    Y_test_obj,
    axis=1
)

y_pred = np.argmax(
    pred,
    axis=1
)

acc = accuracy_score(
    y_true,
    y_pred
)

prec = precision_score(
    y_true,
    y_pred,
    average='weighted'
)

rec = recall_score(
    y_true,
    y_pred,
    average='weighted'
)

f1 = f1_score(
    y_true,
    y_pred,
    average='weighted'
)

print("Accuracy :",acc)
print("Precision:",prec)
print("Recall   :",rec)
print("F1       :",f1)

print(
    classification_report(
        y_true,
        y_pred
    )
)

cnn.save(
    "models/object_classifier.h5"
)

32/32 ━━━━━━━━━━━━━━━━━━━━ 6s 197ms/step


Accuracy : 0.817
Precision: 0.7529739671760046
Recall   : 0.817
F1       : 0.7553915094339623
              precision    recall  f1-score   support

           0       0.82      0.99      0.90       799
           1       0.00      0.00      0.00        13
           2       0.00      0.00      0.00        50
           3       0.77      0.19      0.30       128
           5       0.00      0.00      0.00         9
           6       0.00      0.00      0.00         1

    accuracy                           0.82      1000
   macro avg       0.27      0.20      0.20      1000
weighted avg       0.75      0.82      0.76      1000



# Dữ liệu Lane Detection

In [5]:
IMG_SIZE_SEG = 96
LIMIT_SEG = 300

img_files = sorted(
    os.listdir(SEG_IMG)
)

mask_files = sorted(
    os.listdir(SEG_MASK)
)

X_lane = []
Y_lane = []

for img_name,mask_name in tqdm(
    list(zip(
        img_files,
        mask_files
    ))[:LIMIT_SEG]
):

    img = cv2.imread(
        os.path.join(
            SEG_IMG,
            img_name
        )
    )

    mask = cv2.imread(
        os.path.join(
            SEG_MASK,
            mask_name
        )
    )

    if img is None:
        continue

    if mask is None:
        continue

    img = cv2.resize(
        img,
        (IMG_SIZE_SEG,
         IMG_SIZE_SEG)
    )

    mask = cv2.resize(
        mask,
        (IMG_SIZE_SEG,
         IMG_SIZE_SEG)
    )

    gray = cv2.cvtColor(
        mask,
        cv2.COLOR_BGR2GRAY
    )

    binary = (
        gray > 100
    ).astype(np.float32)

    X_lane.append(img)
    Y_lane.append(binary)

X_lane = np.array(
    X_lane,
    dtype=np.float32
)/255.0

Y_lane = np.array(
    Y_lane,
    dtype=np.float32
)

Y_lane = np.expand_dims(
    Y_lane,
    axis=-1
)

print(X_lane.shape)
print(Y_lane.shape)

100%|██████████| 300/300 [00:09<00:00, 30.32it/s]


(300, 96, 96, 3)
(300, 96, 96, 1)


# U-NET

In [6]:
inputs = tf.keras.Input(
    shape=(96,96,3)
)

c1 = tf.keras.layers.Conv2D(
    16,3,
    activation='relu',
    padding='same'
)(inputs)

c1 = tf.keras.layers.Conv2D(
    16,3,
    activation='relu',
    padding='same'
)(c1)

p1 = tf.keras.layers.MaxPooling2D()(c1)

c2 = tf.keras.layers.Conv2D(
    32,3,
    activation='relu',
    padding='same'
)(p1)

c2 = tf.keras.layers.Conv2D(
    32,3,
    activation='relu',
    padding='same'
)(c2)

u1 = tf.keras.layers.UpSampling2D()(c2)

concat = tf.keras.layers.Concatenate()(
    [u1,c1]
)

outputs = tf.keras.layers.Conv2D(
    1,
    1,
    activation='sigmoid'
)(concat)

unet = tf.keras.Model(
    inputs,
    outputs
)

unet.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_lane = unet.fit(
    X_lane,
    Y_lane,
    epochs=3,
    batch_size=4,
    validation_split=0.2
)

Epoch 1/3
60/60 ━━━━━━━━━━━━━━━━━━━━ 9s 97ms/step - accuracy: 0.7269 - loss: 0.5281 - val_accuracy: 0.7598 - val_loss: 0.5302
Epoch 2/3
60/60 ━━━━━━━━━━━━━━━━━━━━ 10s 93ms/step - accuracy: 0.8224 - loss: 0.4216 - val_accuracy: 0.7937 - val_loss: 0.5037
Epoch 3/3
60/60 ━━━━━━━━━━━━━━━━━━━━ 12s 115ms/step - accuracy: 0.8263 - loss: 0.4048 - val_accuracy: 0.8041 - val_loss: 0.4781


# Đánh giá Lane Detection

In [7]:
pred = unet.predict(X_lane)

pred = (
    pred > 0.5
).astype(np.uint8)

gt = Y_lane.astype(np.uint8)

intersection = np.logical_and(
    gt,
    pred
).sum()

union = np.logical_or(
    gt,
    pred
).sum()

iou = intersection/(union+1e-7)

dice = (
    2*intersection
)/(gt.sum()+pred.sum()+1e-7)

pixel_acc = (
    gt==pred
).mean()

print("IoU:",iou)
print("Dice:",dice)
print(
    "Pixel Accuracy:",
    pixel_acc
)

unet.save(
    "models/lane_detector.h5"
)

10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 203ms/step


IoU: 0.6624295219287675
Dice: 0.7969414801539496
Pixel Accuracy: 0.8292292390046296


# Dữ liệu Video

In [9]:
import cv2
import numpy as np
import tensorflow as tf
from tqdm import tqdm

cnn = tf.keras.models.load_model("models/object_classifier.h5")

unet = tf.keras.models.load_model("models/lane_detector.h5")

CLASS_NAMES = [
    "car",
    "bus",
    "truck",
    "person",
    "rider",
    "bike",
    "motor"
]

VIDEO_PATH = "test_video.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise Exception(
        f"Cannot open {VIDEO_PATH}"
    )

width = int(
    cap.get(cv2.CAP_PROP_FRAME_WIDTH)
)

height = int(
    cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
)

fps = cap.get(
    cv2.CAP_PROP_FPS
)

frame_count = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

OUTPUT_VIDEO = "result.mp4"

fourcc = cv2.VideoWriter_fourcc(
    *'mp4v'
)

writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    fourcc,
    fps,
    (width,height)
)

print(
    f"Processing {frame_count} frames..."
)

for _ in tqdm(range(frame_count)):

    ret, frame = cap.read()

    if not ret:
        break

    original = frame.copy()

    lane_img = cv2.resize(
        frame,
        (96,96)
    )

    lane_img = cv2.cvtColor(
        lane_img,
        cv2.COLOR_BGR2RGB
    )

    lane_img = (
        lane_img.astype(np.float32)
        / 255.0
    )

    lane_pred = unet.predict(
        lane_img[np.newaxis],
        verbose=0
    )[0]

    lane_pred = (
        lane_pred > 0.5
    ).astype(np.uint8)

    lane_pred = cv2.resize(
        lane_pred,
        (width,height)
    )

    lane_overlay = np.zeros_like(
        frame
    )

    lane_overlay[:,:,1] = (
        lane_pred.squeeze()*255
    )

    frame = cv2.addWeighted(
        frame,
        1.0,
        lane_overlay,
        0.4,
        0
    )

    gray = cv2.cvtColor(
        original,
        cv2.COLOR_BGR2GRAY
    )

    blur = cv2.GaussianBlur(
        gray,
        (5,5),
        0
    )

    edges = cv2.Canny(
        blur,
        100,
        200
    )

    contours, _ = cv2.findContours(
        edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours:

        x,y,w,h = cv2.boundingRect(cnt)

        area = w*h

        if area < 3000:
            continue

        crop = original[
            y:y+h,
            x:x+w
        ]

        if crop.size == 0:
            continue

        try:

            crop = cv2.resize(
                crop,
                (96,96)
            )

            crop = cv2.cvtColor(
                crop,
                cv2.COLOR_BGR2RGB
            )

            crop = (
                crop.astype(np.float32)
                /255.0
            )

            pred = cnn.predict(
                crop[np.newaxis],
                verbose=0
            )[0]

            cls = np.argmax(pred)

            conf = np.max(pred)

            if conf < 0.6:
                continue

            label = (
                f"{CLASS_NAMES[cls]}"
                f" {conf:.2f}"
            )

            cv2.rectangle(
                frame,
                (x,y),
                (x+w,y+h),
                (0,255,0),
                2
            )

            cv2.putText(
                frame,
                label,
                (x,y-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0,255,0),
                2
            )

        except:
            pass

    writer.write(frame)

cap.release()
writer.release()

print("\nDone!")
print("Saved:", OUTPUT_VIDEO)

Processing 530 frames...


100%|██████████| 530/530 [33:52<00:00,  3.84s/it]


Done!
Saved: result.mp4
